URL index: [Home](https://zzz.bwh.harvard.edu/luna-walkthrough/) | [Data](https://zzz.bwh.harvard.edu/luna-walkthrough/data/) | [S1. File QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p1/) | [S2. Signal QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p2) | [S3. Staging](https://zzz.bwh.harvard.edu/luna-walkthrough/p3) | [S4. Artifacts](https://zzz.bwh.harvard.edu/luna-walkthrough/p4) | [S5. Analysis](https://zzz.bwh.harvard.edu/luna-walkthrough/p5)

Notebook index: [Index](../00_index.ipynb) | [S1. File QC](../p1/00_index.ipynb) | [S2. Signal QC](../p2/00_index.ipynb) | [S3. Staging](../p3/00_index.ipynb) | [S4. Artifacts](../p4/00_index.ipynb) | [S5. Analysis](../p5/00_index.ipynb)

---

# 2.2. Finding flat/duplicate channels

Walkthrough URL = [https://zzz.bwh.harvard.edu/luna-walkthrough/p2/dupes/](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/dupes/)

In [3]:
import lunapi as lp
proj = lp.proj()
proj.sample_list( '../harm1.lst' )

initiated lunapi v1.2.3 <lunapi.lunapi0.luna object at 0x1339b32f0> 

read 20 individuals from ../harm1.lst


In this section, we will scan for flat or duplicated channels in the EEG, using the `DUPES` command:

In [5]:
# luna harm1.lst -o out.db -s ' DUPES sig=${eeg} '
res = proj.silent_proc( 'DUPES sig=${eeg}' ) 

In [7]:
# destrat out.db +DUPES
proj.table( 'DUPES' )

,ID,DUPES,FLAT,INVALID
0,F01,0,0,None
1,F02,0,0,None
2,F03,0,0,None
3,F04,0,0,None
4,F05,0,0,None
5,F06,0,0,None
6,F07,0,0,None
7,F08,0,0,None
8,F09,0,0,None
9,F10,0,0,None


We see potential issues with two recordings:
 - `M01` has 36 duplicate channel pairs
 - `M03` has 15 duplicate channel pairs

We can see which channels and channel pairs had issues (adding a space between individuals for clarity): 

In [9]:
# destrat out.db +DUPES -r CH
proj.table( 'DUPES' , 'CH' ) 

,ID,CH,DUPE,FLAT
0,M01,AFZ,1,0
1,M03,C3,1,0
2,M03,C4,1,0
3,M01,CPZ,1,0
4,M01,CZ,1,0
5,M03,F3,1,0
6,M03,F4,1,0
7,M01,FCZ,1,0
8,M01,FPZ,1,0
9,M01,FZ,1,0


We can also get a list of the pairwise duplicates as follows: 

In [10]:
# destrat out.db +DUPES -r CHS
proj.table( 'DUPES' , 'CHS' )

,ID,CHS,DUPE
0,M01,"AFZ,CPZ",1
1,M01,"AFZ,CZ",1
2,M01,"AFZ,FCZ",1
3,M01,"AFZ,FPZ",1
4,M01,"AFZ,FZ",1
5,M01,"AFZ,OZ",1
6,M01,"AFZ,POz",1
7,M01,"AFZ,PZ",1
8,M03,"C3,C4",1
9,M03,"C3,F3",1


__Please read the [original page](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/dupes/)__ for context on these results. 

---

We'll re-run DUPES with these two changes in place reflecting the points above: 1) pointing to the original data (`s1.lst` rather than `harm1.lst`) and 2) adding the physical argument:

In [24]:
# luna s1.lst -o out.db -s ' DUPES sig=${eeg} physical '
proj.sample_list( '../s1.lst' )
res = proj.silent_proc( 'DUPES sig=${eeg} physical' ) 

read 20 individuals from ../s1.lst


In [25]:
# destrat out.db +DUPES
proj.table( 'DUPES' )

,ID,DUPES,FLAT,INVALID
0,F01,0,0,None
1,F02,0,0,None
2,F03,0,0,None
3,F04,0,0,None
4,F05,0,0,None
5,F06,0,0,None
6,F07,0,0,None
7,F08,0,0,None
8,F09,0,0,None
9,F10,0,0,None


In [27]:
# destrat out.db +DUPES -r CH 
proj.table( 'DUPES' , 'CH' ).sort_values( 'ID' )

,ID,CH,DUPE,FLAT
0,M01,AFZ,1,1
3,M01,CPZ,1,1
4,M01,CZ,1,1
8,M01,FCZ,1,1
9,M01,FPZ,1,1
10,M01,FZ,1,1
11,M01,OZ,1,1
14,M01,POz,1,1
15,M01,PZ,1,1
1,M03,C3,1,0


That is, as expected, we can now see that:
  - `M01` has all 9 midline channels listed as flat; and naturally, these 9 flat channels are all identically flat, and so we have 9 * 8 / 2 = 36 pairs of "duplicated" channels also
  - `M03` now has 21 (not 15) duplicates pairs, i.e. 7 * 6 / 2 = 21, as we have 7 channels including CZ that are all duplicated


---
We can now move on to the [next section](03_stats.ipynb) where we generate some summary statistics for the records.